In [1]:
# class RobotabilityGraph that inherits from Graph class 
import os
import sys
for _root in (os.path.abspath(".."), os.path.abspath(".")):
    if os.path.isfile(os.path.join(_root, "src", "utils", "logger.py")):
        if _root not in sys.path:
            sys.path.insert(0, _root)
        break

import osmnx as ox 
import geopandas as gpd 
import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt
from src.utils.plotting import setup_matplotlib

from glob import glob 
from tqdm import tqdm 

from shapely import wkt, LineString 

import rasterio
from rasterio.enums import Resampling
from rasterio.plot import show 


from src.utils.logger import setup_logger 

logger = setup_logger('rs-street-furniture')
logger.setLevel("INFO")
if setup_matplotlib(usetex=True):
    logger.info("Matplotlib using LaTeX text.")
else:
    logger.info("Matplotlib LaTeX unavailable (install cm-super to enable); using mathtext.")
logger.info("Modules initialized.")

WGS='EPSG:4326'
PROJ='EPSG:2263'

REGEN_SEGMENTIZATION=False
REGEN_TOPOLOGY=True

GEN_INSPECTION_PLOTS=True
INSPECTION_PLOTS="figures/inspection_plots"

USE_ALTERNATE_GEN_METHOD=True

os.makedirs(INSPECTION_PLOTS, exist_ok=True)


2026-08-17 16:50:11 - rs-street-furniture - INFO - Matplotlib using LaTeX text.
2026-08-17 16:50:11 - rs-street-furniture - INFO - Modules initialized.


## Loading and Preprocessing Data Features 

### Sidewalk Basemap (NYC)

In [2]:


if USE_ALTERNATE_GEN_METHOD:
    segmentized = pd.read_csv("data/sidewalks_nyc_segmentized.csv")
    segmentized = gpd.GeoDataFrame(segmentized, geometry=segmentized['geometry'].apply(wkt.loads), crs=PROJ)
    logger.info("Segmentized sidewalk basemap loaded.")


    sidewalk_nyc = segmentized

    logger.success("NYC sidewalk basemap loaded.")
    logger.info(f"Distribution of sidewalk widths [ft]: \n{sidewalk_nyc['width'].describe()}")
else: 
    segmentized = pd.read_csv("data/sidewalks_nyc_segmentized.csv")
    segmentized = gpd.GeoDataFrame(segmentized, geometry=segmentized['geometry'].apply(wkt.loads), crs=PROJ)
    logger.info("Segmentized sidewalk basemap loaded.")


    sidewalk_nyc = segmentized

    logger.success("NYC sidewalk basemap loaded.")
    logger.info(f"Distribution of sidewalk widths [ft]: \n{sidewalk_nyc['SHAPE_Width'].describe()}")

2026-08-17 16:50:22 - rs-street-furniture - INFO - Segmentized sidewalk basemap loaded.
2026-08-17 16:50:22 - rs-street-furniture - SUCCESS - NYC sidewalk basemap loaded.
2026-08-17 16:50:22 - rs-street-furniture - INFO - Distribution of sidewalk widths [ft]: 
count    1.901084e+06
mean     3.077903e+00
std      1.823185e+00
min      7.731023e-04
25%      2.077818e+00
50%      2.722169e+00
75%      3.645861e+00
max      5.037467e+01
Name: width, dtype: float64


In [3]:
# the maximum distance to search for a nearby street segment. Since we segmentize by 50 feet, we can search within 25 feet
MAX_DISTANCE=25

CUTOFF= pd.to_datetime("2023-08-31")


In [4]:

# we buffer each point by 25 feet, creating a 50-diameter circle centered at the point. This captures nearby clutter. 
sidewalk_nyc['geometry'] = sidewalk_nyc['geometry'].buffer(MAX_DISTANCE)

### Bus Stop Shelters 

In [5]:
# read bus stop shelters 
bus_stop_shelters = gpd.read_file("data/street_furniture/bus_stop_shelters_nyc.csv")
bus_stop_shelters = gpd.GeoDataFrame(bus_stop_shelters, geometry=wkt.loads(bus_stop_shelters['the_geom']), crs=WGS).to_crs(PROJ)

# Bus stop installation date is not present, so filtering is out-of-scoped.

### Trash Cans / Waste Baskets 

In [6]:
# load trash cans 
trash_cans = pd.read_csv("data/street_furniture/dsny_litter_baskets_nyc.csv")
trash_cans = gpd.GeoDataFrame(trash_cans, geometry=wkt.loads(trash_cans['point']), crs=WGS).to_crs(PROJ)

# trash can installation date is not present, so filtering is out-of-scope

### LinkNYC Kiosks 

In [9]:
# load linknyc
# linknyc = pd.read_csv("data/street_furniture/LinkNYC_Kiosk_Locations_20240816.csv")
# Edit @raphael, got it manually from https://data.cityofnewyork.us/Social-Services/LinkNYC-Kiosk-Locations/s4kf-3yrf/about_data and updated path
linknyc = pd.read_csv("data/street_furniture/LinkNYC_Kiosk_Locations_20260817.csv")
linknyc = gpd.GeoDataFrame(linknyc, geometry=gpd.points_from_xy(linknyc['Longitude'], linknyc['Latitude']), crs='EPSG:4326').to_crs(PROJ)

linknyc['Installation Complete'] = pd.to_datetime(linknyc['Installation Complete'])
linknyc = linknyc[linknyc['Installation Complete'] <= CUTOFF]
linknyc['Installation Complete'].describe()

count                             2138
mean     2017-12-17 16:15:56.407857664
min                1971-12-01 00:00:00
25%                2016-11-11 06:00:00
50%                2017-07-18 00:00:00
75%                2018-02-17 00:00:00
max                2023-07-27 00:00:00
Name: Installation Complete, dtype: object

### Bicycle Parking Shelters 

In [10]:
# load bicycle parking shelters 
bicycle_parking_shelters = pd.read_csv("data/street_furniture/bicycle_parking_shelters_nyc.csv")
bicycle_parking_shelters = gpd.GeoDataFrame(bicycle_parking_shelters, geometry=wkt.loads(bicycle_parking_shelters['the_geom']), crs=WGS).to_crs(PROJ)
bicycle_parking_shelters['Build_date'] = pd.to_datetime(bicycle_parking_shelters['Build_date'])
bicycle_parking_shelters = bicycle_parking_shelters[bicycle_parking_shelters['Build_date'] <= CUTOFF]
bicycle_parking_shelters['Build_date'].describe()

/tmp/ipykernel_122168/602664558.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bicycle_parking_shelters['Build_date'] = pd.to_datetime(bicycle_parking_shelters['Build_date'])


count                               17
mean     2008-07-26 18:21:10.588235264
min                2007-12-17 00:00:00
25%                2008-07-01 00:00:00
50%                2008-09-12 00:00:00
75%                2008-10-13 00:00:00
max                2008-12-17 00:00:00
Name: Build_date, dtype: object

### Bicycle Racks 

In [22]:
# load bicycle racks 
bicycle_racks = gpd.read_file("data/street_furniture/Bicycle Parking_20260817/geo_export_8f146c93-5220-484b-a9e5-9efb225527d9.shp").to_crs(PROJ)
print(bicycle_racks.columns)
# bicycle_racks['Date_Inst'] = pd.to_datetime(bicycle_racks['Date_Inst'])
# bicycle_racks = bicycle_racks[bicycle_racks['Date_Inst'] <= CUTOFF]
# bicycle_racks['Date_Inst'].describe()
bicycle_racks['date_date_'] = pd.to_datetime(bicycle_racks['date_date_'])
bicycle_racks = bicycle_racks[bicycle_racks['date_date_'] <= CUTOFF]
bicycle_racks['date_date_'].describe()

Index(['borocode', 'boroname', 'borocd', 'coundist', 'assemdist', 'stsendist',
       'congdist', 'program', 'site_id', 'group_id', 'borough', 'ifoaddress',
       'onstreet', 'fromstreet', 'tostreet', 'side_of_st', 'racktype',
       'date_date_', 'time_date_', 'latitude', 'longitude', 'ntaname',
       'femafldz', 'femafldt', 'hrcevac', 'geometry'],
      dtype='object')


count                            36664
mean     2007-12-16 18:44:04.032293376
min                1900-01-01 00:00:00
25%                2010-05-18 00:00:00
50%                2013-07-17 00:00:00
75%                2018-02-10 00:00:00
max                2023-08-31 00:00:00
Name: date_date_, dtype: object

### CityBench 

In [23]:
# load citybench
citybench = pd.read_csv("data/street_furniture/citybench_nyc.csv")
citybench = gpd.GeoDataFrame(citybench, geometry=gpd.points_from_xy(citybench['Longitude'], citybench['Latitude']), crs='EPSG:4326').to_crs(PROJ)
citybench['Installati'] = pd.to_datetime(citybench['Installati'])
citybench = citybench[citybench['Installati'] <= CUTOFF]
citybench['Installati'].describe()

count                             2141
mean     2015-04-09 03:36:34.301728256
min                2012-04-01 00:00:00
25%                2013-07-10 00:00:00
50%                2014-11-18 00:00:00
75%                2017-02-04 00:00:00
max                2019-09-30 00:00:00
Name: Installati, dtype: object

### Street Trees 

In [24]:
# load trees 
trees = pd.read_csv("data/street_furniture/forestry_tree_points_nyc.csv", engine='pyarrow')
trees = gpd.GeoDataFrame(trees, geometry=gpd.points_from_xy(trees['longitude'], trees['latitude']), crs='EPSG:4326').to_crs(PROJ)
trees['created_at'] = pd.to_datetime(trees['created_at'])
trees = trees[trees['created_at'] <= CUTOFF]
trees['created_at'].describe()

count                           683788
mean     2015-12-06 07:18:59.574254592
min                2015-05-19 00:00:00
25%                2015-08-29 00:00:00
50%                2015-10-23 00:00:00
75%                2016-02-06 00:00:00
max                2016-10-05 00:00:00
Name: created_at, dtype: object

### News Stands 

In [25]:
# load newsstands 
newsstands = pd.read_csv("data/street_furniture/newsstands_nyc.csv", engine='pyarrow')
newsstands = gpd.GeoDataFrame(newsstands, geometry=wkt.loads(newsstands['the_geom']), crs='EPSG:4326').to_crs(PROJ)
newsstands['Built_Date'] = pd.to_datetime(newsstands['Built_Date'])
newsstands = newsstands[newsstands['Built_Date'] <= CUTOFF]
newsstands['Built_Date'].describe() 

count                              357
mean     2011-09-04 23:31:45.882352896
min                2007-09-03 00:00:00
25%                2008-07-30 00:00:00
50%                2010-11-08 00:00:00
75%                2013-06-04 00:00:00
max                2021-12-22 00:00:00
Name: Built_Date, dtype: object

### Parking Meters 

In [26]:
# load parking meters 
parking_meters = pd.read_csv("data/street_furniture/parking_meters_nyc.csv")
parking_meters = gpd.GeoDataFrame(parking_meters, geometry=wkt.loads(parking_meters['Location']), crs='EPSG:4326').to_crs(PROJ)

# parking meter installation date is not present, so filtering is out-of-scope

### Fire Hydrants 

In [27]:
# load hydrants 
hydrants = pd.read_csv("data/street_furniture/fire_hydrants_nyc.csv")
hydrants = gpd.GeoDataFrame(hydrants, geometry=wkt.loads(hydrants['the_geom']), crs='EPSG:4326').to_crs(PROJ)

# hydrant installation date is not present, so filtering is out-of-scope

### Street Signs 

In [29]:
# load street signs 
# street_signs = pd.read_csv("data/street_furniture/Street_Sign_Work_Orders_20240816.csv", engine='pyarrow')
street_signs = pd.read_csv("data/street_furniture/Street_Sign_Work_Orders_20260817.csv", engine='pyarrow')
# only keep 'Current' record type 
street_signs = street_signs[street_signs['record_type'] == 'Current']
street_signs['order_completed_on_date'] = pd.to_datetime(street_signs['order_completed_on_date'])
street_signs = street_signs[street_signs['order_completed_on_date'] <= CUTOFF]
street_signs = gpd.GeoDataFrame(street_signs, geometry=gpd.points_from_xy(street_signs['sign_x_coord'], street_signs['sign_y_coord']), crs=PROJ)
street_signs['order_completed_on_date'].describe()

count                           689352
mean     2018-10-04 16:46:01.006858752
min                1954-09-01 00:00:00
25%                2017-05-31 00:00:00
50%                2019-05-23 00:00:00
75%                2021-01-27 00:00:00
max                2023-08-31 00:00:00
Name: order_completed_on_date, dtype: object

### Bollards 

In [30]:
# load bollards 
# bollards = pd.read_csv("data/street_furniture/Traffic_Bollards_Tracking_and_Installations_20240816.csv", engine='pyarrow')
bollards = pd.read_csv("data/street_furniture/Traffic_Bollards_Tracking_and_Installations_20260817.csv", engine='pyarrow')

bollards['Date'] = pd.to_datetime(bollards['Date'])
bollards = bollards[bollards['Date'] <= CUTOFF]
bollards['Date'].describe()

# we choose not to process bollards, as locations need to be geocoded. Latitude/Longitude is not present in the dataset.

count                            54665
mean     2016-05-17 19:10:59.939632128
min                2005-01-10 00:00:00
25%                2012-09-10 00:00:00
50%                2017-07-12 00:00:00
75%                2020-06-25 00:00:00
max                2023-08-31 00:00:00
Name: Date, dtype: object

### In-Service Fire Alarm Call Boxes 

In [32]:
# alarm_call_boxes = pd.read_csv("data/street_furniture/In-Service_Alarm_Box_Locations_20240816.csv")
alarm_call_boxes = pd.read_csv("data/street_furniture/In-Service_Alarm_Box_Locations_20260817.csv")
alarm_call_boxes = gpd.GeoDataFrame(alarm_call_boxes, geometry=wkt.loads(alarm_call_boxes['Location Point']), crs='EPSG:4326').to_crs(PROJ)

### Scaffolding 

In [40]:
# read DoB active scaffolding permits 
# edit @raphael could not find so took this https://data.cityofnewyork.us/Housing-Development/DOB-NOW-Build-Job-Application-Filings/w9ak-ipjd/about_data
# scaffolding_permits = pd.read_csv("data/dob_active_sheds.csv", engine='pyarrow')
scaffolding_permits = pd.read_csv("data/DOB_NOW__Build___Job_Application_Filings.csv", engine='pyarrow')
scaffolding_permits = gpd.GeoDataFrame(scaffolding_permits, geometry=gpd.points_from_xy(scaffolding_permits['Longitude'], scaffolding_permits['Latitude']), crs='EPSG:4326')
# scaffolding_permits = gpd.GeoDataFrame(scaffolding_permits, geometry=gpd.points_from_xy(scaffolding_permits['Longitude Point'], scaffolding_permits['Latitude Point']), crs='EPSG:4326')
scaffolding_permits = scaffolding_permits.to_crs(PROJ)

Index(['Job Filing Number', 'Filing Status', 'House No', 'Street Name',
       'Borough', 'Block', 'LOT', 'Bin', 'Commmunity - Board', 'Work on Floor',
       'Apt./Condo No(s)', 'Applicant Professional Title',
       'Applicant License #', 'Applicant First Name',
       'Applicants Middle Initial', 'Applicant Last Name',
       'Owner's Business Name', 'Applicant Street Name', 'Applicant City',
       'Applicant State', 'Applicant Zip', 'Filing Representative First Name',
       'Filing Representative Middle Initial',
       'Filing Representative Last Name',
       'Filing Representative Business Name',
       'Filing Representative Street Name', 'Filing Representative City',
       'Filing Representative State', 'Filing Representative Zip',
       'Sprinkler (Work Type)', 'Plumbing (Work Type)',
       'Stand Pipe (Work Type)', 'Antenna (Work Type)', 'Sign (Work Type)',
       'Curb Cut (Work Type)', 'Fence (Work Type)', 'Scaffold (Work Type)',
       'Shed (Work Type)', 'Initial Co

In [41]:
scaffolding_permits['First Permit Date']  = pd.to_datetime(scaffolding_permits['First Permit Date'])

scaffolding_permits = scaffolding_permits[scaffolding_permits['First Permit Date'] <= CUTOFF]
scaffolding_permits['First Permit Date'].describe()

/tmp/ipykernel_122168/3013352855.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  scaffolding_permits['First Permit Date']  = pd.to_datetime(scaffolding_permits['First Permit Date'])


count                        298326
mean     2021-08-16 11:21:00.494784
min             2016-09-20 00:00:00
25%             2020-09-10 00:00:00
50%             2021-11-18 00:00:00
75%             2022-10-08 00:00:00
max             2023-08-31 00:00:00
Name: First Permit Date, dtype: object

## Spatial Joining of Street Furnitures to Sidewalk Graph 

In [42]:
# sjoin nearest bus stops and trash cans to sidewalk
len_before = len(sidewalk_nyc)
bus_stop_shelters = gpd.sjoin(sidewalk_nyc, bus_stop_shelters, )
logger.info(f"Missing {len(bus_stop_shelters[bus_stop_shelters['index_right'].isna()])} bus stop shelters.")

2026-08-17 17:17:35 - rs-street-furniture - INFO - Missing 0 bus stop shelters.


In [43]:
# sjoin nearest trash cans to sidewalk
len_before = len(trash_cans)
trash_cans = gpd.sjoin(sidewalk_nyc, trash_cans, )
logger.info(f"Removed {len_before - len(trash_cans)} trash cans that are not on sidewalks.")

2026-08-17 17:17:37 - rs-street-furniture - INFO - Removed -2136 trash cans that are not on sidewalks.


In [44]:
# sjoin nearest linknyc to sidewalk
len_before = len(linknyc)
linknyc = gpd.sjoin(sidewalk_nyc, linknyc, )
logger.info(f"LinkNYC: {len_before} -> {len(linknyc)}")

2026-08-17 17:17:39 - rs-street-furniture - INFO - LinkNYC: 2138 -> 5509


In [45]:
# sjoin nearest citybench 
len_before = len(citybench)
citybench = gpd.sjoin(sidewalk_nyc, citybench, )
logger.info(f"Citybench: {len_before} -> {len(citybench)}")

2026-08-17 17:17:40 - rs-street-furniture - INFO - Citybench: 2141 -> 775


In [46]:
# sjoint nearest bicycle parking shelters to sidewalk
len_before = len(bicycle_parking_shelters)
bicycle_parking_shelters = gpd.sjoin(sidewalk_nyc, bicycle_parking_shelters, )
logger.info(f"Bicycle Parking Shelters: {len_before} -> {len(bicycle_parking_shelters)}")

2026-08-17 17:17:41 - rs-street-furniture - INFO - Bicycle Parking Shelters: 17 -> 28


In [47]:

# sjoin nearest bicycle racks to sidewalk
len_before = len(bicycle_racks)
bicycle_racks = gpd.sjoin(sidewalk_nyc, bicycle_racks, )
logger.info(f"Bicycle Racks: {len_before} -> {len(bicycle_racks)}")

2026-08-17 17:17:43 - rs-street-furniture - INFO - Bicycle Racks: 36664 -> 55363


In [48]:
# sjoin nearest trees to sidewalk
len_before = len(trees)
trees = gpd.sjoin(sidewalk_nyc, trees, )
logger.info(f"Trees: {len_before} -> {len(trees)}")

2026-08-17 17:17:48 - rs-street-furniture - INFO - Trees: 683788 -> 894352


In [49]:
# sjoin nearest newsstands to sidewalk
len_before = len(newsstands)
newsstands = gpd.sjoin(sidewalk_nyc, newsstands, )
logger.info(f"Newsstands: {len_before} -> {len(newsstands)}")

2026-08-17 17:17:49 - rs-street-furniture - INFO - Newsstands: 357 -> 488


In [50]:
# sjoin nearest parking meters to sidewalk
len_before = len(parking_meters)
parking_meters = gpd.sjoin(sidewalk_nyc, parking_meters, )
logger.info(f"Parking Meters: {len_before} -> {len(parking_meters)}")

2026-08-17 17:17:50 - rs-street-furniture - INFO - Parking Meters: 15598 -> 19149


In [51]:
# sjoin nearest hydrants to sidewalk
len_before = len(hydrants)
hydrants = gpd.sjoin(sidewalk_nyc, hydrants, )
logger.info(f"Hydrants: {len_before} -> {len(hydrants)}")

2026-08-17 17:17:52 - rs-street-furniture - INFO - Hydrants: 109725 -> 167597


In [52]:
# sjoin nearest street signs to sidewalk
len_before = len(street_signs)
street_signs = gpd.sjoin(sidewalk_nyc, street_signs, )
logger.info(f"Street Signs: {len_before} -> {len(street_signs)}")

2026-08-17 17:17:57 - rs-street-furniture - INFO - Street Signs: 689352 -> 489854


In [53]:
# sjoin nearest bollards to sidewalk
#len_before = len(bollards)
#bollards = gpd.sjoin(sidewalk_nyc, bollards )
#logger.info(f"Bollards: {len_before} -> {len(bollards)}")


In [54]:
# sjoin nearest alarm call boxes to sidewalk
len_before = len(alarm_call_boxes)
alarm_call_boxes = gpd.sjoin(sidewalk_nyc, alarm_call_boxes, )
logger.info(f"Alarm Call Boxes: {len_before} -> {len(alarm_call_boxes)}")

2026-08-17 17:17:59 - rs-street-furniture - INFO - Alarm Call Boxes: 13008 -> 8462


In [55]:
BUFFER=100 
# buffer scaffolding_permits points, then sjoin to sidewalks
scaffolding_permits.geometry = scaffolding_permits.geometry.buffer(BUFFER)
scaffolding_permits = gpd.sjoin(sidewalk_nyc, scaffolding_permits, predicate='intersects')

In [56]:

# now, get number of bus stops, trash cans, linknyc, citybench, bicycle parking shelters, and bicycle racks per sidewalk
bus_stop_counts = bus_stop_shelters.groupby('point_index').size().reset_index(name='bus_stop_count').fillna(0)
trash_can_counts = trash_cans.groupby('point_index').size().reset_index(name='trash_can_count').fillna(0)
linknyc_counts = linknyc.groupby('point_index').size().reset_index(name='linknyc_count').fillna(0)
citybench_counts = citybench.groupby('point_index').size().reset_index(name='citybench_count').fillna(0)
bicycle_parking_shelter_counts = bicycle_parking_shelters.groupby('point_index').size().reset_index(name='bicycle_parking_shelter_count').fillna(0)
bicycle_rack_counts = bicycle_racks.groupby('point_index').size().reset_index(name='bicycle_rack_count').fillna(0)
tree_counts = trees.groupby('point_index').size().reset_index(name='tree_count').fillna(0)
newsstand_counts = newsstands.groupby('point_index').size().reset_index(name='newsstand_count').fillna(0)
parking_meter_counts = parking_meters.groupby('point_index').size().reset_index(name='parking_meter_count').fillna(0)
hydrant_counts = hydrants.groupby('point_index').size().reset_index(name='hydrant_count').fillna(0)
street_sign_counts = street_signs.groupby('point_index').size().reset_index(name='street_sign_count').fillna(0)
#bollard_counts = bollards.groupby('point_index').size().reset_index(name='bollard_count').fillna(0)
alarm_call_box_counts = alarm_call_boxes.groupby('point_index').size().reset_index(name='alarm_call_box_count').fillna(0)

In [57]:
# merge scaffolding in 
scaffolding_counts = scaffolding_permits.groupby('point_index').size().reset_index(name='scaffolding_permit_count').fillna(0)

In [58]:

# merge counts to sidewalk_nyc
sidewalk_nyc = sidewalk_nyc.merge(bus_stop_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(trash_can_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(linknyc_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(citybench_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(bicycle_parking_shelter_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(bicycle_rack_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(tree_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(newsstand_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(parking_meter_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(hydrant_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(street_sign_counts, on='point_index', how='left')
#sidewalk_nyc = sidewalk_nyc.merge(bollard_counts, on='point_index', how='left')
sidewalk_nyc = sidewalk_nyc.merge(alarm_call_box_counts, on='point_index', how='left')

In [59]:
# merge scaffolding in 
sidewalk_nyc = sidewalk_nyc.merge(scaffolding_counts, on='point_index', how='left')

In [60]:

sidewalk_nyc = sidewalk_nyc.fillna(0)

In [61]:
sidewalk_nyc.describe([0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975, 0.99])

,Unnamed: 0,width,segment_index,point_index,bus_stop_count,trash_can_count,linknyc_count,citybench_count,bicycle_parking_shelter_count,bicycle_rack_count,tree_count,newsstand_count,parking_meter_count,hydrant_count,street_sign_count,alarm_call_box_count,scaffolding_permit_count
count,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06,1.901084e+06
mean,9.505415e+05,3.077903e+00,2.384651e+05,9.505415e+05,2.770525e-03,1.186113e-02,2.897820e-03,4.076622e-04,1.472844e-05,2.912181e-02,4.704432e-01,2.566957e-04,1.007267e-02,8.815865e-02,2.576709e-01,4.451145e-03,2.983136e+00
std,5.487958e+05,1.823185e+00,1.387391e+05,5.487958e+05,5.328835e-02,1.104973e-01,5.375337e-02,3.325726e-02,3.837738e-03,2.456756e-01,7.140006e-01,1.608521e-02,1.003709e-01,2.854762e-01,8.370054e-01,6.831537e-02,9.528991e+00
min,0.000000e+00,7.731023e-04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1%,1.901083e+04,7.063126e-01,4.631830e+03,1.901083e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2.5%,4.752708e+04,9.565545e-01,1.140200e+04,4.752708e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
5%,9.505415e+04,1.204814e+00,2.266515e+04,9.505415e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
10%,1.901083e+05,1.514769e+00,4.455830e+04,1.901083e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.752708e+05,2.077818e+00,1.181800e+05,4.752708e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,9.505415e+05,2.722169e+00,2.377430e+05,9.505415e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [62]:
# naive weights based on predicted area of different clutters 
weights = { 
    'bus_stop_count': 2,
    'trash_can_count': 0.5, 
    'linknyc_count': 2, 
    'citybench_count': 1.5,
    'bicycle_parking_shelter_count': 2,
    'bicycle_rack_count': 1.5,
    'tree_count': .15,
    'newsstand_count': 3, 
    'parking_meter_count': .15,
    'scaffolding_permit_count': 2,
    'hydrant_count': 0.25,
    'street_sign_count': 0.05,
    #'bollard_count': 0.05,
    'alarm_call_box_count': .15
}

In [63]:

# create a 'clutter' metric that is the sum of all street clutter features
sidewalk_nyc['clutter'] = 0
for feature, weight in weights.items():
    sidewalk_nyc['clutter'] += sidewalk_nyc[feature] * weight

sidewalk_nyc['clutter'].describe([0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975, 0.99])

count    1.901084e+06
mean     6.136301e+00
std      1.912672e+01
min      0.000000e+00
1%       0.000000e+00
2.5%     0.000000e+00
5%       0.000000e+00
10%      0.000000e+00
25%      0.000000e+00
50%      2.500000e-01
75%      5.150000e+00
90%      1.600000e+01
95%      2.805000e+01
97.5%    4.495000e+01
99%      7.800000e+01
max      8.460000e+02
Name: clutter, dtype: float64

In [64]:
# Now, weighted clutter by sidewalk width 
if USE_ALTERNATE_GEN_METHOD:
    sidewalk_nyc['clutter'] = sidewalk_nyc['clutter'] / sidewalk_nyc['width']
else: 
    sidewalk_nyc['clutter'] = sidewalk_nyc['clutter'] / sidewalk_nyc['SHAPE_Width']

In [65]:
# clamp distribution to 1st and 99th percentile
sidewalk_nyc['clutter'] = sidewalk_nyc['clutter'].clip(lower=sidewalk_nyc['clutter'].quantile(0.01), upper=sidewalk_nyc['clutter'].quantile(0.99))

In [66]:
# final describe 
sidewalk_nyc['clutter'].describe([0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975, 0.99])

count    1.901084e+06
mean     1.832237e+00
std      4.027790e+00
min      0.000000e+00
1%       0.000000e+00
2.5%     0.000000e+00
5%       0.000000e+00
10%      0.000000e+00
25%      0.000000e+00
50%      9.065140e-02
75%      1.798733e+00
90%      5.317316e+00
95%      9.297551e+00
97.5%    1.494301e+01
99%      2.508611e+01
max      2.508616e+01
Name: clutter, dtype: float64

In [67]:
# write street furniture density to csv 
os.makedirs("data/processed", exist_ok=True)
sidewalk_nyc.to_csv("data/processed/street_furniture_density.csv", index=False)

In [ ]:
# map sidewalk and color by clutter 
fig, ax = plt.subplots(figsize=(20, 20))
sidewalk_nyc.plot(column='clutter', ax=ax, legend=True, cmap='cividis', markersize=0.25, legend_kwds={'label': "Weighted Street Clutter", 'orientation': 'horizontal', 'shrink': 0.5, 'pad': 0.01})
ax.set_axis_off()

plt.savefig("figures/street_furniture_density.png", dpi=150, bbox_inches='tight', pad_inches=0)
plt.close()

: 